In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load clean data
candidates = pd.read_csv('../data/clean/candidates_clean.csv', parse_dates=['application_date'])
offers     = pd.read_csv('../data/clean/offers_clean.csv', parse_dates=['offer_date', 'join_date'])
pipeline   = pd.read_csv('../data/clean/pipeline_events_clean.csv', parse_dates=['event_date'])
recruiters = pd.read_csv('../data/clean/recruiters_clean.csv')

print(f"Candidates: {len(candidates):,}")
print(f"Offers:     {len(offers):,}")
print(f"Pipeline:   {len(pipeline):,}")
print("✅ Data loaded")

Candidates: 1,800
Offers:     179
Pipeline:   5,649
✅ Data loaded


In [2]:
# ── TEST 1: Two-Sample T-Test ─────────────────────────────────
# Question: Is time-to-hire significantly different between
#           LinkedIn and non-LinkedIn sourced candidates?
# Why t-test: We're comparing a continuous variable (days)
#             between two independent groups.

# Merge candidates with accepted offers only
accepted = offers[offers['accepted'] == True].copy()
merged = candidates.merge(accepted, on='candidate_id')
merged['days_to_hire'] = (merged['offer_date'] - merged['application_date']).dt.days

# Split into two groups
linkedin     = merged[merged['source_channel'] == 'LinkedIn']['days_to_hire'].dropna()
non_linkedin = merged[merged['source_channel'] != 'LinkedIn']['days_to_hire'].dropna()

# Run the t-test
t_stat, p_value = stats.ttest_ind(linkedin, non_linkedin)

print("── T-Test: LinkedIn vs Non-LinkedIn Time-to-Hire ──────────")
print(f"LinkedIn     → n={len(linkedin)},  median={linkedin.median():.1f} days,  mean={linkedin.mean():.1f} days")
print(f"Non-LinkedIn → n={len(non_linkedin)}, median={non_linkedin.median():.1f} days,  mean={non_linkedin.mean():.1f} days")
print(f"\nt-statistic: {t_stat:.3f}")
print(f"p-value:     {p_value:.4f}")
print()
if p_value < 0.05:
    print("✅ STATISTICALLY SIGNIFICANT (p < 0.05)")
    print("   The difference in TTH between LinkedIn and non-LinkedIn")
    print("   candidates is unlikely to be due to random chance.")
else:
    print("⚠️  NOT STATISTICALLY SIGNIFICANT (p >= 0.05)")
    print("   We cannot conclude the channels differ in time-to-hire.")

── T-Test: LinkedIn vs Non-LinkedIn Time-to-Hire ──────────
LinkedIn     → n=40,  median=31.0 days,  mean=40.2 days
Non-LinkedIn → n=70, median=33.5 days,  mean=42.8 days

t-statistic: -0.676
p-value:     0.5005

⚠️  NOT STATISTICALLY SIGNIFICANT (p >= 0.05)
   We cannot conclude the channels differ in time-to-hire.


In [3]:
# ── TEST 2: Chi-Square Test ───────────────────────────────────
# Question: Is rejection reason distribution independent of
#           which recruiter made the decision?
# Why chi-square: We're testing independence between two
#                 categorical variables.

failures = pipeline[
    (pipeline['outcome'] == 'fail') &
    (pipeline['reject_reason'] != 'Not Provided')
].merge(recruiters[['recruiter_id', 'name']], on='recruiter_id')

# Build contingency table: rows = recruiters, columns = reasons
contingency = pd.crosstab(failures['name'], failures['reject_reason'])

print("Contingency table (recruiter × rejection reason):")
print(contingency)
print()

# Run chi-square
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

print(f"── Chi-Square Test Results ─────────────────────────────────")
print(f"Chi-square statistic: {chi2:.2f}")
print(f"Degrees of freedom:   {dof}")
print(f"p-value:              {p_value:.4f}")
print()
if p_value < 0.05:
    print("✅ STATISTICALLY SIGNIFICANT")
    print("   Rejection reasons are NOT independent of recruiter.")
    print("   Certain recruiters reject for different reasons — possible bias.")
else:
    print("⚠️  NOT STATISTICALLY SIGNIFICANT (p >= 0.05)")
    print("   Rejection reasons are distributed similarly across recruiters.")
    print("   The bottleneck is speed, not judgement differences.")

Contingency table (recruiter × rejection reason):
reject_reason      Culture mismatch  Failed technical assessment  \
name                                                               
Anay Shanker                     28                           32   
Aniruddh Batra                   23                           36   
Jhanvi Chaudhary                 17                           34   
Kiara Kakar                      26                           43   
Madhup Kapur                     14                           33   
Mehul Krishnan                   20                           33   
Nirvaan Choudhury                21                           40   
Sara Behl                        19                           44   

reject_reason      Insufficient experience  No-show  Role cancelled  \
name                                                                  
Anay Shanker                            38       24              11   
Aniruddh Batra                          32       16     

In [6]:
# ── TEST 3: Pearson Correlation (Fixed) ───────────────────────

# Force event_date to datetime to be safe
pipeline['event_date'] = pd.to_datetime(pipeline['event_date'])

# Step 1: Calculate TTH per recruiter per candidate (in days as integer)
rp = pipeline.merge(recruiters[['recruiter_id', 'name']], on='recruiter_id')

tth_rows = []
for (name, cid), grp in rp.groupby(['name', 'candidate_id']):
    days = (grp['event_date'].max() - grp['event_date'].min()).days
    tth_rows.append({'name': name, 'tth_days': days})

tth_df = pd.DataFrame(tth_rows)

# Step 2: Median TTH per recruiter
median_tth = tth_df.groupby('name')['tth_days'].median().rename('median_tth')

# Step 3: Acceptance rate per recruiter
rp2 = pipeline.merge(recruiters[['recruiter_id', 'name']], on='recruiter_id') \
              .merge(offers[['candidate_id', 'accepted']], on='candidate_id', how='left')
acceptance = rp2.groupby('name')['accepted'].mean().rename('acceptance_rate')

# Step 4: Combine
summary = pd.concat([median_tth, acceptance], axis=1).dropna()
summary['median_tth'] = summary['median_tth'].astype(float)
summary['acceptance_rate'] = summary['acceptance_rate'].astype(float)

print("Per-recruiter summary:")
print(summary.round(3))
print()

# Step 5: Correlation
corr, p_value = stats.pearsonr(summary['median_tth'], summary['acceptance_rate'])

print(f"── Pearson Correlation: Recruiter Speed vs Acceptance Rate ─")
print(f"Correlation coefficient: {corr:.3f}")
print(f"p-value:                 {p_value:.4f}")
print()
if corr < -0.3:
    print("📉 Negative correlation: slower recruiters → lower acceptance rates")
elif corr > 0.3:
    print("📈 Positive correlation: slower recruiters → higher acceptance rates")
else:
    print("➡️  Weak/no linear relationship between recruiter speed and acceptance")

if p_value < 0.05:
    print("✅ Statistically significant")
else:
    print("⚠️  Not statistically significant — only 8 recruiters, small sample")

Per-recruiter summary:
                   median_tth  acceptance_rate
name                                          
Anay Shanker              8.0            0.750
Aniruddh Batra            8.0            0.773
Jhanvi Chaudhary         24.0            0.731
Kiara Kakar              10.0            0.619
Madhup Kapur              8.0            0.429
Mehul Krishnan            8.0            0.500
Nirvaan Choudhury         8.0            0.625
Sara Behl                20.0            0.478

── Pearson Correlation: Recruiter Speed vs Acceptance Rate ─
Correlation coefficient: 0.046
p-value:                 0.9131

➡️  Weak/no linear relationship between recruiter speed and acceptance
⚠️  Not statistically significant — only 8 recruiters, small sample


## Statistical Findings Summary

### Test 1 — T-Test: LinkedIn vs Non-LinkedIn TTH
[Write here: was the result significant? What were the medians?
What does this mean for sourcing strategy?]

### Test 2 — Chi-Square: Recruiter vs Rejection Reason
[Write here: were rejection reasons independent of recruiter?
What does this mean — is the problem speed or judgement?]

### Test 3 — Correlation: Recruiter Speed vs Acceptance Rate
[Write here: direction of correlation, strength, what it suggests
about whether slow recruiters are losing candidates at offer stage]

## Key Business Implication
[Write 2-3 sentences connecting all three tests back to the
original question: why did offer acceptance decline?]